# 115 — Semantic Kernel: Microsoft's AI Orchestration SDK
## What you'll learn: plugins, kernel functions, and auto function calling
⏱ ~45 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/115-semantic-kernel/semantic_kernel_workbook.ipynb)

Microsoft Semantic Kernel (SK) is the production AI orchestration SDK powering Microsoft 365 Copilot, Azure AI Studio, and dozens of enterprise deployments. Instead of defining an explicit graph (LangGraph), SK uses **plugins** — collections of `@kernel_function`-decorated methods — that the LLM can call automatically.

This workshop teaches the SK mental model: Kernel → Plugin → KernelFunction → FunctionChoiceBehavior.Auto.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — SK architecture: Kernel, Plugin, KernelFunction |
| 2 | **Setup** — install deps, configure API key |
| 3 | **Plugins** — creating @kernel_function decorated methods |
| 4 | **Kernel** — registering plugins and services |
| 5 | **Auto function calling** — letting the LLM decide the sequence |
| 6 | **SK vs LangGraph** — mental model comparison |
| 7 | **Full pipeline** — end-to-end demo |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- `semantic-kernel`

### Key References
> [Semantic Kernel GitHub](https://github.com/microsoft/semantic-kernel)
>
> [SK Python docs](https://learn.microsoft.com/en-us/semantic-kernel/get-started/quick-start-guide?pivots=programming-language-python)
>
> [SK Plugin architecture](https://learn.microsoft.com/en-us/semantic-kernel/concepts/plugins/)

## Part 1 — Concepts: The Semantic Kernel Mental Model

### The core abstraction

```
LangGraph mental model:
  You define an explicit directed graph.
  Nodes are functions. Edges are routes.
  The developer controls the execution order.

Semantic Kernel mental model:
  You register plugins (collections of tools).
  The LLM decides which tools to call and when.
  The Kernel is an orchestrator, not a graph.
```

### The three core objects

```
Kernel
  ├── Services       (LLM providers: OpenAI, Azure OpenAI, etc.)
  └── Plugins
        └── KernelFunctions  (@kernel_function decorated methods)
```

**Kernel**: the central orchestrator. Holds references to AI services and plugins.

**Plugin**: a Python class whose methods are decorated with `@kernel_function`. SK discovers these automatically when you call `kernel.add_plugin(MyPlugin(), plugin_name="MyPlugin")`.

**KernelFunction**: a single callable skill. The decorator adds metadata (name, description) that SK uses to build the tool definition sent to the LLM.

### FunctionChoiceBehavior.Auto

When you invoke the kernel with `FunctionChoiceBehavior.Auto`, SK:
1. Passes all registered plugin functions to the LLM as tool definitions
2. The LLM decides which functions to call and in what order
3. SK executes the chosen functions and feeds results back to the LLM
4. The loop repeats until the LLM returns a final text response

This is the same mechanism as OpenAI function calling — SK wraps it in a consistent API.

### Why SK exists

SK was built for enterprise settings where:
- Multiple plugins need to be composed without hardcoding the pipeline
- The same plugins must work across Azure OpenAI, OpenAI, HuggingFace backends
- Non-developer "planners" need to define multi-step workflows
- Memory, prompt templates, and personas need to be managed centrally

## Part 2 — Setup

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "semantic-kernel==1.28.0",
         "python-dotenv"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local — skipping install (using requirements.txt)")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 3 — Creating Plugins with @kernel_function

### What @kernel_function does

The decorator registers metadata on the method:
- `name`: the function name the LLM sees in the tool definition
- `description`: the docstring shown to the LLM — this is the most important field

A clear description is essential. The LLM selects functions based on description alone.

### Plugin anatomy

```python
class MyPlugin:
    @kernel_function(
        name="my_function",
        description="Does X when given Y — be specific here",
    )
    def my_function(self, input: str) -> str:
        return f"result: {input}"
```

SK discovers all `@kernel_function` methods when you call `kernel.add_plugin(MyPlugin(), ...)`.
You can add as many methods as needed to a single plugin class.

In [ ]:
from semantic_kernel.functions import kernel_function


class WebSearchPlugin:
    """Simulates web search results for common topics."""

    SEARCH_RESULTS = {
        "climate change": (
            "Climate change refers to long-term shifts in global temperatures and weather "
            "patterns. Since the 1800s, human activities — mainly burning fossil fuels — "
            "have been the main driver. CO2 levels have risen 50% above pre-industrial "
            "levels. The IPCC reports that limiting warming to 1.5C requires net-zero "
            "emissions by 2050."
        ),
        "large language models": (
            "Large language models (LLMs) are neural networks trained on vast text corpora "
            "using self-supervised learning. GPT-4, Claude, and Gemini are current leaders. "
            "Key capabilities: text generation, summarization, code synthesis, and reasoning. "
            "Emergent behaviors appear at scale above 100B parameters."
        ),
        "renewable energy": (
            "Renewable energy sources — solar, wind, hydro, geothermal — account for 30% "
            "of global electricity in 2024. Solar PV costs fell 90% from 2010-2023. "
            "IEA projects renewables will supply 60% of electricity by 2030."
        ),
    }

    @kernel_function(
        name="search",
        description="Search the web for information on any topic. Returns relevant facts.",
    )
    def search(self, query: str) -> str:
        query_lower = query.lower()
        for key, result in self.SEARCH_RESULTS.items():
            if key in query_lower:
                return f"[Search results for '{query}']\n{result}"
        return f"[Search results for '{query}']\nGeneral information not available in demo data."


class SummarizerPlugin:
    """Summarizes and extracts key points from text."""

    @kernel_function(
        name="summarize",
        description="Condense a long text into a shorter 2-3 sentence summary.",
    )
    def summarize(self, text: str) -> str:
        parts = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
        return ". ".join(parts[:2]) + "." if parts else text

    @kernel_function(
        name="extract_key_points",
        description="Extract the 3 most important points from a text as a bullet list.",
    )
    def extract_key_points(self, text: str) -> str:
        parts = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
        bullets = [f"- {p}" for p in parts[:3]]
        return "\n".join(bullets)


# Verify the functions have metadata
print("WebSearchPlugin.search description:")
print(f"  {WebSearchPlugin.search.__sk_function_metadata__.description}")
print("\nSummarizerPlugin.summarize description:")
print(f"  {SummarizerPlugin.summarize.__sk_function_metadata__.description}")

## Part 4 — Building the Kernel

### The Kernel as dependency injector

The Kernel acts as a service locator and plugin registry:

```
kernel = sk.Kernel()
kernel.add_service(OpenAIChatCompletion(service_id="default", ai_model_id="gpt-4o-mini"))
kernel.add_plugin(WebSearchPlugin(), plugin_name="WebSearch")
kernel.add_plugin(SummarizerPlugin(), plugin_name="Summarizer")
```

When you invoke the kernel, it:
1. Resolves the correct AI service
2. Discovers available plugin functions
3. Passes them as tool definitions to the LLM

### Plugin namespacing

Plugin functions are referenced as `PluginName.FunctionName`:
- `WebSearch.search`
- `Summarizer.summarize`
- `Summarizer.extract_key_points`

This namespacing prevents function name collisions when multiple plugins
define a function with the same name (e.g., both having a `search` method).

In [ ]:
import semantic_kernel as sk
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

kernel = sk.Kernel()

# Register the AI service
kernel.add_service(
    OpenAIChatCompletion(
        service_id="default",
        ai_model_id="gpt-4o-mini",
    )
)

# Register plugins
kernel.add_plugin(WebSearchPlugin(), plugin_name="WebSearch")
kernel.add_plugin(SummarizerPlugin(), plugin_name="Summarizer")

# Inspect what's registered
print("Registered services:", list(kernel.services.keys()))
print("Registered plugins:", list(kernel.plugins.keys()))
print()
print("Available functions:")
for plugin_name, plugin in kernel.plugins.items():
    for func_name in plugin:
        func = plugin[func_name]
        print(f"  {plugin_name}.{func_name}: {func.description}")

## Part 5 — Auto Function Calling

### FunctionChoiceBehavior.Auto

This is the SK equivalent of OpenAI's `tools` + `tool_choice="auto"`. When you set:

```python
settings = OpenAIChatPromptExecutionSettings(
    function_choice_behavior=FunctionChoiceBehavior.Auto(...)
)
```

The kernel:
1. Serializes all registered functions as JSON Schema tool definitions
2. Passes them to the LLM with the conversation history
3. If the LLM calls a function: SK executes it, adds the result to history, loops
4. If the LLM returns text: SK returns the final response

### The filter parameter

`FunctionChoiceBehavior.Auto(filters={"included_plugins": ["WebSearch", "Summarizer"]})`

Without filters, all registered plugins are available. Filters let you scope which
plugins are offered for a specific invocation — useful when you have many plugins
but only some are relevant to the current task.

In [ ]:
import asyncio
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings
from semantic_kernel.contents.chat_history import ChatHistory


async def run_research_task(kernel: sk.Kernel, goal: str) -> str:
    """Run a research goal through the kernel with auto function calling."""
    settings = OpenAIChatPromptExecutionSettings(
        function_choice_behavior=FunctionChoiceBehavior.Auto(
            filters={"included_plugins": ["WebSearch", "Summarizer"]}
        )
    )

    history = ChatHistory()
    history.add_system_message(
        "You are a research assistant. Use WebSearch to find information, "
        "then use Summarizer to condense the findings into a clear response."
    )
    history.add_user_message(goal)

    chat_service = kernel.get_service("default")
    result = await chat_service.get_chat_message_content(
        chat_history=history,
        settings=settings,
        kernel=kernel,
    )
    return str(result)


# Run the first research task
goal = "Search for information about climate change and give me a brief summary."
print(f"Goal: {goal}\n")
response = await run_research_task(kernel, goal)
print(f"Response:\n{response}")

In [ ]:
# Run two more goals to see different plugin sequences

goals = [
    "Find out about large language models and extract the key points.",
    "Search for renewable energy information and summarize it in 2 sentences.",
]

for i, g in enumerate(goals, 1):
    print(f"\n{'='*65}")
    print(f"Goal {i}: {g}")
    print("-"*65)
    resp = await run_research_task(kernel, g)
    print(f"Response: {resp[:400]}")

## Part 6 — SK vs LangGraph: Mental Model Comparison

This is the core teaching point of the example. Both achieve tool use, but with different philosophy.

### Execution model

```
LangGraph (explicit graph):
  Developer defines:
    - nodes (functions)
    - edges (transitions)
    - conditional edges (routing logic)
  Result: deterministic execution order, visible as a graph

Semantic Kernel (implicit orchestration):
  Developer defines:
    - plugins (tool collections)
    - kernel (the orchestrator)
  LLM decides:
    - which tools to call
    - in what order
    - whether to loop
  Result: emergent execution order, driven by LLM reasoning
```

### Comparison table

| Dimension | LangGraph | Semantic Kernel |
|-----------|-----------|-----------------|
| Execution control | Developer | LLM |
| Graph visibility | Explicit (you draw it) | Implicit (hidden in LLM) |
| Debugging | Deterministic, traceable | Probabilistic, harder to trace |
| Flexibility | High (arbitrary logic) | High (LLM chooses freely) |
| Predictability | High | Medium (LLM-dependent) |
| Enterprise features | Checkpointing, streaming | Planner, Memory, Personas |
| Best for | Multi-step pipelines, RAG | Copilot-style assistants |
| Microsoft ecosystem | No | Yes (Azure OpenAI first-class) |

### When to choose SK

- Building a Microsoft 365 / Azure OpenAI integration
- Need built-in memory, persona, and prompt template management
- Want the LLM to plan tool sequences dynamically
- Team prefers "register and forget" over "draw the graph"

### When to choose LangGraph

- Need deterministic, auditable pipelines
- Building RAG with strict retrieval steps
- Multi-agent systems with explicit handoffs
- Need streaming, interrupt-resume, or checkpointing

In [ ]:
# Side-by-side: how the same task looks in SK vs LangGraph pseudocode

SK_CODE = '''
# Semantic Kernel approach
kernel = sk.Kernel()
kernel.add_plugin(WebSearchPlugin(), "WebSearch")
kernel.add_plugin(SummarizerPlugin(), "Summarizer")

# LLM decides whether to call search, summarize, or both
result = await kernel.invoke_with_auto_function_calling(goal)
'''

LANGGRAPH_CODE = '''
# LangGraph approach
graph = StateGraph(ResearchState)
graph.add_node("search", search_node)
graph.add_node("summarize", summarize_node)
graph.add_edge(START, "search")          # always search first
graph.add_edge("search", "summarize")   # always summarize after
graph.add_edge("summarize", END)
app = graph.compile()
result = app.invoke({"goal": goal})
'''

print("Semantic Kernel:")
print(SK_CODE)
print("\nLangGraph:")
print(LANGGRAPH_CODE)
print()
print("Key difference: SK lets the LLM decide the sequence.")
print("LangGraph makes the sequence explicit and deterministic.")

## Part 7 — Full Pipeline: main.py End-to-End

This replicates what `main.py` does from the command line.

In [ ]:
async def run_full_demo() -> None:
    print("=== 115 — Semantic Kernel: Plugins and Auto Function Calling ===\n")

    # Build kernel
    demo_kernel = sk.Kernel()
    demo_kernel.add_service(
        OpenAIChatCompletion(service_id="default", ai_model_id="gpt-4o-mini")
    )
    demo_kernel.add_plugin(WebSearchPlugin(), plugin_name="WebSearch")
    demo_kernel.add_plugin(SummarizerPlugin(), plugin_name="Summarizer")

    print(f"Plugins registered: {list(demo_kernel.plugins.keys())}")
    print(f"Functions available: {sum(len(p) for p in demo_kernel.plugins.values())}\n")

    # Research tasks
    tasks = [
        "Search for climate change information and give me a 2-sentence summary.",
        "Find what you know about large language models and list 3 key points.",
        "What is the current state of renewable energy? Search and summarize.",
    ]

    for i, task in enumerate(tasks, 1):
        print(f"{'='*65}")
        print(f"Task {i}: {task}")
        print("-"*65)
        resp = await run_research_task(demo_kernel, task)
        print(f"Response:\n{resp[:400]}")
        print()

    print("="*65)
    print("\nThe Kernel sequenced WebSearch -> Summarizer automatically.")
    print("No explicit graph definition was needed.")


await run_full_demo()

## Part 8 — Inspecting Function Call Sequences

A common debugging need: which functions did the LLM actually call, and in what order?
SK exposes this through the `inner_content` of the response message.

In [ ]:
async def run_with_trace(kernel: sk.Kernel, goal: str) -> dict:
    """Run a goal and capture the full chat history including tool calls."""
    settings = OpenAIChatPromptExecutionSettings(
        function_choice_behavior=FunctionChoiceBehavior.Auto(
            filters={"included_plugins": ["WebSearch", "Summarizer"]}
        )
    )

    history = ChatHistory()
    history.add_system_message(
        "You are a research assistant. Use WebSearch then Summarizer to answer."
    )
    history.add_user_message(goal)

    chat_service = kernel.get_service("default")
    result = await chat_service.get_chat_message_content(
        chat_history=history,
        settings=settings,
        kernel=kernel,
    )

    # Count tool calls in the history
    tool_calls = [
        msg for msg in history.messages
        if hasattr(msg, "role") and str(msg.role) in ("tool", "function")
    ]

    return {
        "goal": goal,
        "response": str(result),
        "history_length": len(history.messages),
        "tool_calls_count": len(tool_calls),
    }


trace = await run_with_trace(kernel, "Search for climate change and extract key points.")
print(f"Goal: {trace['goal'][:60]}...")
print(f"History messages: {trace['history_length']}")
print(f"Tool call messages: {trace['tool_calls_count']}")
print(f"\nFinal response:\n{trace['response'][:400]}")

## Exercises

### Exercise 1 — Add a third plugin

Create a `FactCheckerPlugin` with a single `@kernel_function` method `verify(claim: str) -> str` that:
- Returns `"VERIFIED"` if the claim contains words like "CO2", "LLM", "solar", or "wind"
- Returns `"UNVERIFIED"` otherwise

Register it on the kernel and ask a goal that should trigger all 3 plugins.

---

### Exercise 2 — Restricted function choice

Use `FunctionChoiceBehavior.Auto(filters={"included_plugins": ["WebSearch"]})` to restrict
the kernel to only the WebSearch plugin. Run the same climate change goal. How does the
response differ when the LLM cannot access Summarizer?

---

### Exercise 3 — Compare SK and raw tool calling

Implement the same research task using the raw OpenAI Python client with:
- `tools=[{"type": "function", "function": {"name": "search", "description": "...", "parameters": {...}}}]`
- Manual tool call execution

Compare the code length and structure to the SK version. What does SK add?

In [ ]:
# ===== ANSWER KEY — Exercise 1: Third plugin =====

class FactCheckerPlugin:
    """Verifies whether a claim matches known facts in the demo data."""

    KNOWN_FACTS = {"co2", "llm", "solar", "wind", "climate", "energy", "model", "neural"}

    @kernel_function(
        name="verify",
        description="Check whether a factual claim can be verified against known information.",
    )
    def verify(self, claim: str) -> str:
        words = set(claim.lower().split())
        if words & self.KNOWN_FACTS:
            return f"VERIFIED: The claim contains known facts: {words & self.KNOWN_FACTS}"
        return "UNVERIFIED: Claim does not match known fact patterns in the demo database."


# Add to kernel and test
kernel.add_plugin(FactCheckerPlugin(), plugin_name="FactChecker")

print("FactChecker plugin registered.")
print(f"Total plugins: {list(kernel.plugins.keys())}")

# Quick unit test
fc = FactCheckerPlugin()
print(f"\nverify('CO2 levels are rising'): {fc.verify('CO2 levels are rising')}")
print(f"verify('The moon is made of cheese'): {fc.verify('The moon is made of cheese')}")

In [ ]:
# ===== ANSWER KEY — Exercise 2: Restricted function choice =====

async def run_with_restriction(kernel, goal, allowed_plugins):
    settings = OpenAIChatPromptExecutionSettings(
        function_choice_behavior=FunctionChoiceBehavior.Auto(
            filters={"included_plugins": allowed_plugins}
        )
    )
    history = ChatHistory()
    history.add_system_message("You are a research assistant. Answer using available tools.")
    history.add_user_message(goal)

    chat_service = kernel.get_service("default")
    result = await chat_service.get_chat_message_content(
        chat_history=history, settings=settings, kernel=kernel
    )
    return str(result)


goal = "Search for climate change information and give a 2-sentence summary."

print("With WebSearch + Summarizer (all plugins):")
resp_all = await run_with_restriction(kernel, goal, ["WebSearch", "Summarizer"])
print(f"  {resp_all[:200]}...")

print("\nWith WebSearch ONLY (Summarizer restricted):")
resp_search_only = await run_with_restriction(kernel, goal, ["WebSearch"])
print(f"  {resp_search_only[:200]}...")

print("\nDifference: without Summarizer, the LLM writes its own summary from raw search output.")

In [ ]:
# ===== ANSWER KEY — Exercise 3: Compare SK to raw tool calling =====

from openai import OpenAI

openai_client = OpenAI()

# Raw OpenAI tool calling (no SK)
RAW_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the web for information on a topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"}
                },
                "required": ["query"],
            },
        },
    }
]

def raw_search(query: str) -> str:
    """Simulate the same search without SK."""
    return (
        "Climate change refers to long-term shifts in global temperatures. "
        "CO2 levels have risen 50% above pre-industrial levels."
    )

# Step 1: initial call
messages = [
    {"role": "system", "content": "Use the search tool to answer."},
    {"role": "user", "content": "What is climate change?"},
]
r1 = openai_client.chat.completions.create(model="gpt-4o-mini", tools=RAW_TOOLS, messages=messages)

# Step 2: execute tool calls manually
if r1.choices[0].message.tool_calls:
    tc = r1.choices[0].message.tool_calls[0]
    tool_result = raw_search("climate change")
    messages.append(r1.choices[0].message)
    messages.append({"role": "tool", "content": tool_result, "tool_call_id": tc.id})

    # Step 3: get final response
    r2 = openai_client.chat.completions.create(model="gpt-4o-mini", messages=messages)
    raw_response = r2.choices[0].message.content
else:
    raw_response = r1.choices[0].message.content

print("Raw OpenAI tool calling response:")
print(f"  {raw_response[:200]}")
print()
print(f"Lines of manual tool-loop code: ~15")
print(f"Lines of SK equivalent code:    ~8  (FunctionChoiceBehavior.Auto handles the loop)")
print()
print("SK abstracts: tool definition serialization, result injection, loop detection.")

## Workshop Complete

You have explored Microsoft Semantic Kernel's core abstractions:

- **`@kernel_function`** — decorates Python methods as AI-callable tools with name + description
- **Plugin classes** — group related functions; registered with `kernel.add_plugin()`
- **Kernel** — the central orchestrator; holds services (LLMs) and plugins (tools)
- **FunctionChoiceBehavior.Auto** — delegates tool selection and sequencing to the LLM
- **Plugin filters** — scope which functions are offered for a specific invocation

**Key insight**: SK and LangGraph both achieve tool use, but with opposite philosophies. SK is implicit (LLM chooses), LangGraph is explicit (developer defines the graph). Neither is universally better — SK excels at open-ended copilot tasks; LangGraph excels at deterministic pipelines.

---

Next: **example 116** — LlamaIndex ReAct Agent with QueryEngineTool over a document collection.

---
### Further reading
- [Semantic Kernel Python docs](https://learn.microsoft.com/en-us/semantic-kernel/get-started/quick-start-guide?pivots=programming-language-python)
- [SK Plugin concepts](https://learn.microsoft.com/en-us/semantic-kernel/concepts/plugins/)
- [FunctionChoiceBehavior reference](https://github.com/microsoft/semantic-kernel/blob/main/python/semantic_kernel/connectors/ai/function_choice_behavior.py)</cell id="cell-md-023"></cell>
